# The reproduction number

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/10-reproduction-number.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

The **reproduction number** is the average number of secondary infectious people
produced by one infectious person. We distinguish:

- $R_0$ — the basic reproduction number in a fully susceptible population
- $R_t$ — the time-varying (effective) reproduction number during the epidemic

The contact rate $\beta$ is secondary infections **per unit time**; $R_0$ is
secondary infections **per infectious case**. For a simple frequency-dependent
SIR with recovery rate $\gamma$ and no competing exits from infectious,

$$
R_0 = \frac{\beta}{\gamma}, \qquad \beta = R_0 \, \gamma.
$$

```{admonition} What summer4 does not compute for you
:class: note

There is no public next-generation-matrix / spectral-radius helper on
`FlowModel` / `ForceOfInfection`. For stratified or multi-compartment infectious states you would
derive $R_0$ yourself. This chapter stays with an **unstratified** SIR and the
closed-form $R_t = R_0 \, S(t)/N$, which matches the source textbook's simple
model.
```


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

AXIS_I = {"index": "time", "value": "number infectious"}


def make_y0(pmap: PropertyMap, state: Property, population: float, seed: float) -> np.ndarray:
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["susceptible"])] = population - seed
    y0[pmap.select(state["infectious"])] = seed
    return y0


def infectious_series(res: Any, state: Property) -> pd.Series:
    return res["comp"].select(state["infectious"]).to_pandas().iloc[:, 0]


def recovered_series(res: Any, state: Property) -> pd.Series:
    return res["comp"].select(state["recovered"]).to_pandas().iloc[:, 0]


def susceptible_series(res: Any, state: Property) -> pd.Series:
    return res["comp"].select(state["susceptible"]).to_pandas().iloc[:, 0]


def build_sir_model() -> tuple[Any, PropertyMap, Property]:
    """Unstratified SIR (dummy pop + unit mixing) without infection death."""
    state = Property("state", ("susceptible", "infectious", "recovered"))
    pop = Property("pop", ("all",))
    pmap = PropertyMap.from_property(state).stratify(pop)
    model = FlowModel(pmap)
    
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind="frequency",
    contact_rate=Param("contact_rate"),
    mixing=mixing,
),
    )
)
    model.add_flow(TransitionFlow(
        "recovery",
        state["infectious"],
        state["recovered"],
        Param("recovery"),
    ))
    return model.compile(), pmap, state


model_config = {"population": 1000.0, "seed": 10.0, "end_time": 20.0}
recovery = 0.333
sojourn_infectious = 1.0 / recovery
cm, pmap, state = build_sir_model()


We fix recovery and set `contact_rate` from the $R_0$ we want to explore
($\beta = R_0 / T_I$ with $T_I = 1/\gamma$).


In [ ]:
def get_output_from_r0s(
    *,
    basic_reproduction_numbers: tuple[float, ...] | np.ndarray,
    population: float,
    seed: float,
    end_time: float,
    compartment: str,
) -> pd.DataFrame:
    """Run the SIR model for each R0 and return the named compartment series."""
    compiled, pmap_local, state_local = build_sir_model()
    times = np.linspace(0.0, end_time, int(end_time * 10) + 1)
    plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
    y0 = make_y0(pmap_local, state_local, population, seed)
    columns: dict[float, pd.Series] = {}
    for r0 in basic_reproduction_numbers:
        params = {"recovery": recovery, "contact_rate": float(r0) / sojourn_infectious}
        res = compiled.run(params, y0, t0=0.0, t1=end_time, dt=0.1, save=plan, solver="dopri5")
        if compartment == "infectious":
            columns[float(r0)] = infectious_series(res, state_local)
        elif compartment == "recovered":
            columns[float(r0)] = recovered_series(res, state_local)
        else:
            raise ValueError(compartment)
    return pd.DataFrame(columns)


## Effect of $R_0$

Higher $R_0$ means a steeper early exponential take-off and earlier susceptible
depletion, so the peak arrives sooner as well as higher.


In [ ]:
high_r0s = (2.0, 3.0, 5.0, 10.0)
high = get_output_from_r0s(
    basic_reproduction_numbers=high_r0s,
    population=model_config["population"],
    seed=model_config["seed"],
    end_time=20.0,
    compartment="infectious",
)
assert float(high[10.0].max()) > float(high[2.0].max())
assert high[10.0].idxmax() < high[2.0].idxmax()
high.plot(labels=AXIS_I, title="Infectious prevalence by R0")


## Threshold value for epidemic take-off

$R_0 = 1$ is the tipping point: above it the epidemic grows, below it infection
declines. Deterministic ODEs do not include stochastic fade-out. Values very
near one are slightly shifted because a non-zero seed already reduces the
initial susceptible fraction below one.


In [ ]:
close_1_r0s = (0.9, 0.95, 1.05, 1.1)
near = get_output_from_r0s(
    basic_reproduction_numbers=close_1_r0s,
    population=model_config["population"],
    seed=model_config["seed"],
    end_time=100.0,
    compartment="infectious",
)
assert float(near[1.1].iloc[-1]) > float(near[0.9].iloc[-1])
assert float(near[0.9].iloc[-1]) < model_config["seed"]
near.plot(labels=AXIS_I, title="Near the R0 = 1 threshold")


## Epidemic final size

Larger $R_0$ infects a larger share of the population by the end of a long run
(approaching, but not reaching, 100% as $R_0$ grows large).


In [ ]:
final_size_r0s = np.linspace(1.0, 4.0, 31)
final = get_output_from_r0s(
    basic_reproduction_numbers=final_size_r0s,
    population=1.0,
    seed=0.01,
    end_time=200.0,
    compartment="recovered",
)
final_size = final.iloc[-1]
assert float(final_size.iloc[-1]) > float(final_size.iloc[0])
final_size.plot(
    labels={"index": "R0", "value": "epidemic final size"},
    title="Final size vs R0",
).update_layout(showlegend=False)


## The time-varying reproduction number, $R_t$

At the start of an outbreak in a fully susceptible population, $R_t \approx R_0$.
As susceptibles are depleted,

$$
R_t = R_0 \times \frac{S(t)}{N}.
$$

When $S/N$ falls below $1/R_0$, we expect $R_t < 1$ and the epidemic to turn
over. (The source notebook uses a constant contact rate derived from $R_0$;
summer4's `timevarying` helpers are not required here.)


In [ ]:
r0 = 4.0
population = 10.0
seed = 0.1
end_time = 20.0
params = {"recovery": recovery, "contact_rate": r0 / sojourn_infectious}
compiled, pmap_rt, state_rt = build_sir_model()
times = np.linspace(0.0, end_time, int(end_time * 10) + 1)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
y0 = make_y0(pmap_rt, state_rt, population, seed)
res = compiled.run(params, y0, t0=0.0, t1=end_time, dt=0.1, save=plan, solver="dopri5")

s = susceptible_series(res, state_rt)
i = infectious_series(res, state_rt)
rt = r0 * s / population
frame = pd.DataFrame(
    {
        "reproduction_number": rt,
        "infectious": i,
        "threshold": np.ones_like(times),
    },
    index=times,
)
# Peak infectiousness should occur near Rt crossing one.
cross = float(frame.index[frame["reproduction_number"] < 1.0][0])
peak_i = float(frame["infectious"].idxmax())
assert abs(cross - peak_i) < 2.0
assert abs(float(frame["reproduction_number"].iloc[0]) - r0 * (population - seed) / population) < 1e-5
frame.plot(
    labels={"index": "time", "value": "number or prevalence"},
    title="Rt and infectious prevalence",
)



With a latent period (SEIR), $R_t$ falling through one and the infectious peak
need not coincide exactly — incidence into infectiousness is delayed. We do not
recompute a next-generation $R_0$ for that structure here.

## Herd immunity threshold

Herd immunity is reached when $R_t = 1$. For this closed SIR with constant $N$,

$$
R_0 \times \frac{S}{N} = 1 \quad\Rightarrow\quad
\frac{S}{N} = \frac{1}{R_0} \quad\Rightarrow\quad
\frac{R}{N} \approx 1 - \frac{1}{R_0}
$$

(using $R$ for the recovered share, not $R_t$). That is the classic herd-immunity
threshold calculation.


In [ ]:
hit = 1.0 - 1.0 / r0
print(f"Herd immunity threshold (recovered share) for R0={r0}: {hit}")
assert abs(hit - 0.75) < 1e-12
# By the infectious peak, recovered+infectious should have crossed the HIT order of magnitude.
final_immune_share = float((population - s.iloc[-1]) / population)
assert final_immune_share > hit
